https://github.com/dakoalesia1-boop/ML-fundamentals-2026

## Task: Task Ordering

The project has the following ML pipeline structure:
1. Load and explore the data
2. Identify target variable
3. Handle missing values
4. Encode categorical variables
5. Scale numerical features
6. Split into training and testing sets
7. Train and evaluate logistic regression
8. Handle class imbalance using class weights and SMOTE

The ordering chosen for the pipeline is important to avoid data leakage. If feature scaling or encoding were done before splitting the data, information from the test set could influence the preprocessing. By splitting first and then fitting transformers only on the training data, we preserve the integrity of the evaluation.

Handling missing values before encoding ensures that the encoding methods don’t misinterpret implicit missing strings as legitimate categories.

## Task: Data Loading and Exploration

In [19]:
import pandas as pd

df = pd.read_csv("archive/bank-additional-full.csv", sep=';')
df_info = df.info()
df_head = df.head()

print(df_info)
df_head

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TASK 1: DATA LOADING AND TARGET IDENTIFICATION
try:
    df = pd.read_csv('bank-additional-full.csv', sep=';')
    print("Dataset loaded successfully!")
    
    # Identify the Prediction Target
    target = 'y'
    print(f"🎯 The prediction target is: '{target}' (Term Deposit Subscription)")

    # Initial Data Exploration
    print("\n--- Dataset Dimensions ---")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

    print("\n--- Target Class Distribution ---")
    print(df[target].value_counts())
    
    print("\n--- Percentage Distribution ---")
    print(df[target].value_counts(normalize=True) * 100)

except FileNotFoundError:
    print("Error: 'bank-additional-full.csv' not found in the current directory.")
except Exception as e:
    print(f"An error occurred: {e}")

df.head()

Error: 'bank-additional-full.csv' not found in the current directory.


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## Task: Identify Target Variable

In [21]:
print("Shape of dataset:", df.shape)
print("\nNumerical summary:\n", df.describe())
print("\nCategorical summary:")
print(df.describe(include=['object']))
print("\nTarget value counts (y):")
print(df['y'].value_counts())

Shape of dataset: (41188, 21)

Numerical summary:
                age      duration      campaign         pdays      previous  \
count  41188.00000  41188.000000  41188.000000  41188.000000  41188.000000   
mean      40.02406    258.285010      2.567593    962.475454      0.172963   
std       10.42125    259.279249      2.770014    186.910907      0.494901   
min       17.00000      0.000000      1.000000      0.000000      0.000000   
25%       32.00000    102.000000      1.000000    999.000000      0.000000   
50%       38.00000    180.000000      2.000000    999.000000      0.000000   
75%       47.00000    319.000000      3.000000    999.000000      0.000000   
max       98.00000   4918.000000     56.000000    999.000000      7.000000   

       emp.var.rate  cons.price.idx  cons.conf.idx     euribor3m   nr.employed  
count  41188.000000    41188.000000   41188.000000  41188.000000  41188.000000  
mean       0.081886       93.575664     -40.502600      3.621291   5167.035911  
std

/var/folders/h4/7pjbxh8x1h9gq7v_y0xy6qrw0000gn/T/ipykernel_62499/1391142332.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include=['object']))


## Task: Managing Missing Values

I identifed the implicit missing values encoded as ‘unknown’ in categorical features and will handle these in feature engineering later. 

**Alternative Approach**: 
- Dropping rows with missing values was a simple strategy that I considered. However, since the missing values were widespread in some features removing them could bias the training set or remove important data. 
- Using advanced imputation was strategy I rejected because it is complex and offers minimal expected gain. 

In [22]:
unknown_counts = (df == "unknown").sum().sort_values(ascending=False)

print("Number of 'unknown' values per column:\n")
print(unknown_counts)

Number of 'unknown' values per column:

default           8597
education         1731
housing            990
loan               990
job                330
marital             80
age                  0
poutcome             0
nr.employed          0
euribor3m            0
cons.conf.idx        0
cons.price.idx       0
emp.var.rate         0
duration             0
previous             0
pdays                0
campaign             0
day_of_week          0
month                0
contact              0
y                    0
dtype: int64


In [5]:
unknown_cols = unknown_counts[unknown_counts > 0]
print("Columns with 'unknown' values:\n", unknown_cols)

Columns with 'unknown' values:
 default      8597
education    1731
housing       990
loan          990
job           330
marital        80
dtype: int64


In [ ]:
import numpy as np
# Replace "unknown" with NaN
df.replace("unknown", np.nan, inplace=True)

missing_after = df.isna().sum().sort_values(ascending=False)

print("Missing values per column after replacing 'unknown' with NaN:\n")
print(missing_after)

Missing values per column after replacing 'unknown' with NaN:

default           8597
education         1731
housing            990
loan               990
job                330
marital             80
age                  0
poutcome             0
nr.employed          0
euribor3m            0
cons.conf.idx        0
cons.price.idx       0
emp.var.rate         0
duration             0
previous             0
pdays                0
campaign             0
day_of_week          0
month                0
contact              0
y                    0
dtype: int64


## Handling Imputation

**Imputation strategy:**
- For categorical variables assign the missing values with the most frequent category
- For numerical variables assign the missing values with median values
- Exclude the duration variable in real predictive pipelines 

In [7]:
from sklearn.impute import SimpleImputer
cat_cols = df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(exclude=['object']).columns

cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer = SimpleImputer(strategy='median')

df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
df[num_cols] = num_imputer.fit_transform(df[num_cols])
print(df.isna().sum().sum(), "missing values remain")

0 missing values remain


/var/folders/h4/7pjbxh8x1h9gq7v_y0xy6qrw0000gn/T/ipykernel_62499/3091480320.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


## Task: Encoding Categorical Variables

One-Hot Encoding used to create a binary column for each category level in order for the categorical features to be turned into numeric representations so that the ML model can understand it. Set handle_unknown = 'ignore' so that unseen categories in the future data don't cause errors. 

**Alternative Approaches:**
- Label encoding was rejected because it imposes arbitrary ordinal relationships on nominal categories, which may mislead linear models like Logistic Regression.
- Target encoding was rejected because it risks target leakage in cross-validation.

In [8]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", cat_cols)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# One-Hot Encoder
onehot = OneHotEncoder(handle_unknown='ignore')
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', onehot, cat_cols)
    ],
    remainder='passthrough'
)

encoded_df = preprocessor.fit_transform(df)
print("Encoded feature shape:", encoded_df.shape)

Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome', 'y']
Encoded feature shape: (41188, 59)


/var/folders/h4/7pjbxh8x1h9gq7v_y0xy6qrw0000gn/T/ipykernel_62499/4114757190.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


## Task: Feature Scaling

I will scale numerical features so that they're on similar ranges to improve stability and performance. 
- OneHotEncoder is used for categorial columns
- StandardScaler is used for numerical columns

In [14]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

feature_df = df.drop(columns=['y'])
cat_cols = feature_df.select_dtypes(include=['object']).columns.tolist()
num_cols = feature_df.select_dtypes(exclude=['object']).columns.tolist()
print("Final categorical columns (no target):", cat_cols)
print("Final numeric columns:", num_cols)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)
print("Preprocessing pipeline constructed without target column.")

Final categorical columns (no target): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Final numeric columns: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Preprocessing pipeline constructed without target column.


/var/folders/h4/7pjbxh8x1h9gq7v_y0xy6qrw0000gn/T/ipykernel_62499/2289651496.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = feature_df.select_dtypes(include=['object']).columns.tolist()


## Task: Data Splitting

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

# Separate X and y 
X = df.drop(columns=['y'])
y = df['y'].apply(lambda val: 1 if val == 'yes' else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
log_reg = LogisticRegression(max_iter=1000)
model_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor_scaled),
    ('classifier', log_reg)
])
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.98      0.95      7310
           1       0.71      0.43      0.54       928

    accuracy                           0.92      8238
   macro avg       0.82      0.70      0.74      8238
weighted avg       0.91      0.92      0.91      8238



## Task: Training a Logistic Regression Model

The "yes" class is under-represented meaning the data set has class imbalance. To fix this, I need to first adjust the weights in logistic regression and then resample it using SMOTE. 

**Logistic Regression:**

Logistic regression is a linear classifier that assumes a linear relationship between features and the log-odds of the positive class. It performs well when features are scaled and binary-encoded, and its coefficients are interpretable. Because this assignment focuses on data preparation quality rather than exploring model architectures, logistic regression provides a strong baseline with straightforward assumptions. Scaling ensures that features remain comparable, and class weights help counter imbalance inherent in the dataset.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

log_reg_balanced = LogisticRegression(class_weight='balanced', max_iter=1000)
model_balanced = Pipeline([
    ('preprocess', preprocessor_scaled),
    ('classifier', log_reg_balanced)
])
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print("Balanced Logistic Regression Performance:\n")
print(classification_report(y_test, y_pred_balanced))

Balanced Logistic Regression Performance:

              precision    recall  f1-score   support

           0       0.99      0.86      0.92      7310
           1       0.45      0.91      0.60       928

    accuracy                           0.86      8238
   macro avg       0.72      0.88      0.76      8238
weighted avg       0.93      0.86      0.88      8238



## Task: Addressing Class Imbalance

To address class imbalance both class weighting and SMOTE were used. 

Class weighting automatically tells the model to adjust the cost of misclassifying each class. This approach is a cost sensitive learning approach that works well with linear models and penalizes errors on the minority class more heavilty. 

SMOTE oversampling generates syntetic minority class samples by interpolating between existing minority examples which helps balance the training data. This method avoids data leakage into the test set and helps the model learn a more robust decision boundary for the minority class. 

**Alternative Approaches:**
- Tree-based Model was a more expressive approach that could've been used but its complexity makes this approach outside of the scope as it focuses on model comparison more than data preparation which was the task. 
- Another approach was undersampling instead of oversampling. Undersampling reduces the dataset size risking to remove valuable examples. SMOTE on the other hand balances the dataset without dropping information.

**Potential Pitfalls:**

Multicollinearity: Some numerical features are highly correlated with each other, which could affect coefficient stability and interpretability.

Duration Information Leakage: Duration should not be used in ture predictive pipelines. In a real predictive setting this value would only be known after the call is completed which creates data leakage. The duration is useful for analysis, but in a production model it should be excluded to ensure prediction validity and generalization. 

In [17]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

smote = SMOTE(random_state=42)
smote_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor_scaled),
    ('smote', smote),
    ('classifier', LogisticRegression(max_iter=1000))
])
smote_pipeline.fit(X_train, y_train)
y_pred_smote = smote_pipeline.predict(X_test)

print("SMOTE Logistic Regression Performance:\n")
print(classification_report(y_test, y_pred_smote))


SMOTE Logistic Regression Performance:

              precision    recall  f1-score   support

           0       0.99      0.86      0.92      7310
           1       0.46      0.91      0.61       928

    accuracy                           0.87      8238
   macro avg       0.72      0.89      0.77      8238
weighted avg       0.93      0.87      0.89      8238



Both approaches resulted in similar recall and F1 score on the minority class. Class weighting ajdusts the loss function so the model focuses on minority class errors more, while SMOTE synthesizes new minority class examples before training. Since both models focus heavily on minority, the results are similar with SMOTE showing a slight improvement at the cost of additional preprocessing. 

## Task: Feature Selection

In [18]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

log_reg_fs = LogisticRegression(max_iter=1000, class_weight='balanced')
selector = RFE(estimator=log_reg_fs, n_features_to_select=10, step=1)
selector = selector.fit(
    preprocessor_scaled.fit_transform(X_train), 
    y_train
)
selected_mask = selector.support_
feature_names = preprocessor_scaled.get_feature_names_out()
selected_features = [f for f, selected in zip(feature_names, selected_mask) if selected]

print("Selected Features:", selected_features)

Selected Features: ['cat__month_aug', 'cat__month_jun', 'cat__month_mar', 'cat__month_may', 'cat__month_nov', 'cat__poutcome_success', 'num__duration', 'num__emp.var.rate', 'num__cons.price.idx', 'num__euribor3m']


Applying RFE with logistic regression to identify a subset of the most important features by recursively fiting the model and removing the least important feautre. This helps reduce overfitting and improve model interpretability. 

**Conclusion:**

The logistic regression model with class weights and SMOTE produced competitive recall and F1 scores on the minority class, suggesting that class imbalance was effectively handled.

The feature selection performed, aids interpretability and may support further model development.

Future work could explore nonlinear models and robust cross-validation strategies to further improve performance and generalizability.